# ⛏️ 04. Khai phá Luật Kết hợp với Apriori

**Thành viên thực hiện:** Thành viên 1 (ARM Lead)

Mục tiêu:
- Áp dụng thuật toán Apriori trên tập các trận Thắng (`Win` transactions).
- Tìm tập mục phổ biến (Frequent Itemsets) với các ngưỡng min_support khác nhau.
- Sinh các luật kết hợp (Association Rules) dạng: $\{Tướng_A, TrangBị_X\} \Rightarrow Tướng_B$.
- Đánh giá qua các độ đo: Support, Confidence, Lift (lọc Lift > 1.5).

In [3]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
from src.arm_miner import AssociationRuleMiner
import warnings
warnings.filterwarnings('ignore')

## 1. Nạp ma trận nhị phân (Binary Matrices)

In [4]:
# Nạp dữ liệu nhị phân (ưu tiên nạp pickle, tự động chuyển sang csv nếu lệch phiên bản numpy)
pkl_path = '../data/processed/bin_win.pkl'
csv_path = '../data/processed/bin_win.csv'

try:
    bin_win = pd.read_pickle(pkl_path)
    bin_lose = pd.read_pickle('../data/processed/bin_lose.pkl')
except Exception as e:
    print(f'Nạp pickle gặp cảnh báo ({e}), chuyển sang đọc file CSV an toàn.')
    bin_win = pd.read_csv(csv_path)
    bin_lose = pd.read_csv('../data/processed/bin_lose.csv')

# Lọc tập trung vào Tướng và Trang bị (loại bỏ các trait trùng lặp gây bùng nổ tổ hợp)
target_cols = [c for c in bin_win.columns if c.startswith('Champ_') or c.startswith('Item_')]
bin_win = bin_win[target_cols]
bin_lose = bin_lose[[c for c in bin_lose.columns if c in target_cols]]

print(f'Ma trận Win (Tướng & Trang bị): {bin_win.shape}')
print(f'Ma trận Lose (Tướng & Trang bị): {bin_lose.shape}')

Ma trận Win shape: (2000, 94)
Ma trận Lose shape: (2000, 95)


## 2. Chạy thuật toán Apriori trên tập Thắng (Win)

In [ ]:
# Khởi tạo Miner với max_len=3 (tìm tổ hợp 2-3 tướng/trang bị, chuẩn chiến thuật TFT)
miner = AssociationRuleMiner(min_support=0.03, min_confidence=0.5, min_lift=1.2, max_len=3)

t0 = time.time()
frequent_itemsets, rules, timing = miner.run_apriori(bin_win, win_label=True)
apriori_time = time.time() - t0

print(f'⏱️ Thời gian chạy Apriori: {apriori_time:.2f} giây')
print(f'📦 Số tập mục phổ biến tìm được: {len(frequent_itemsets)}')
print(f'📋 Số luật kết hợp sinh ra: {len(rules)}')

## 3. Khám phá các Tập mục Phổ biến (Frequent Itemsets)

In [ ]:
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
print(frequent_itemsets.groupby('length').size())
frequent_itemsets.sort_values(by='support', ascending=False).head(10)

## 4. Lọc và Phân tích các Luật có Lift > 1.5 (Cộng hưởng thực sự)

In [ ]:
high_lift_rules = miner.filter_high_lift_rules(rules, min_lift=1.5)
print(f'Số luật có Lift > 1.5: {len(high_lift_rules)}')
miner.format_rules(high_lift_rules, top_n=15)

## 5. Trực quan hóa Phân phối Support - Confidence - Lift

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='viridis', alpha=0.7, edgecolors='k')
plt.colorbar(scatter, label='Lift')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules (Apriori - Win Games): Support vs Confidence (màu theo Lift)')
plt.axvline(0.03, color='gray', linestyle='--')
plt.axhline(0.5, color='gray', linestyle='--')
plt.show()

## 6. Nhận xét kết quả Apriori
- Apriori sinh ra nhiều tổ hợp synergy mạnh với Lift > 1.5 (ví dụ: các cặp tướng kích hoạt tộc/hệ chính).
- Hạn chế: Thời gian sinh ứng viên (candidate generation) tương đối lâu khi giảm min_support.